# Scalable Batch Inference with Ray Data

This notebook demonstrates how to perform efficient batch inference on large datasets using Ray Data. You'll learn how to:

- Process datasets too large to fit in memory
- Build CPU preprocessing → GPU inference pipelines
- Use heterogeneous compute for optimal throughput
- Handle images, tabular data, and text at scale

## When to Use Ray Data

- Processing millions of images for classification/embedding
- Running inference on large tabular datasets
- Preprocessing data before training
- ETL pipelines with ML components

## Prerequisites

- 1+ GPUs
- Python 3.9+

## 1. Installation

In [ ]:
!pip install -q "ray[data]" torch torchvision pandas pyarrow pillow scikit-learn

## 2. Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")

from utils import (
    print_gpu_status,
    detect_gpus,
    init_ray,
    shutdown_ray,
    ClusterMode,
)

In [ ]:
print_gpu_status()

gpu_info = detect_gpus()
NUM_GPUS = gpu_info["count"] if gpu_info["available"] else 0

In [ ]:
# Configuration
CLUSTER_MODE = ClusterMode.LOCAL
OUTPUT_PATH = "./runs/batch_inference"

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
init_ray(mode=CLUSTER_MODE)

---

# Part 1: Image Classification at Scale

Process thousands of images through a pretrained model.

## 1.1 Create Sample Dataset

In [ ]:
import ray
import numpy as np
from PIL import Image
import io

# Create synthetic image dataset
# In production, use ray.data.read_images() for real images
def generate_random_images(n_images=1000):
    """Generate random RGB images for demo."""
    images = []
    for i in range(n_images):
        # Create random 224x224 RGB image
        img_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        images.append({"id": i, "image": img_array})
    return images

# Create Ray Dataset
image_data = generate_random_images(1000)
image_ds = ray.data.from_items(image_data)

print(f"Created dataset with {image_ds.count()} images")
print(f"Schema: {image_ds.schema()}")

## 1.2 Define Model Predictor

In [ ]:
import torch
from torchvision import models, transforms
from typing import Dict


class ImageClassifier:
    """
    Stateful image classifier using pretrained ResNet.
    
    Ray Data instantiates this once per GPU actor,
    so the model is loaded only once and reused.
    """
    
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Load pretrained model
        self.model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.model = self.model.to(self.device)
        self.model.eval()
        
        # Preprocessing transform
        self.transform = transforms.Compose([
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])
        
        # Load ImageNet class labels
        self.labels = models.ResNet50_Weights.DEFAULT.meta["categories"]
    
    def __call__(self, batch: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
        """
        Process a batch of images.
        
        Args:
            batch: Dictionary with 'image' key containing numpy arrays
        
        Returns:
            Dictionary with predictions added
        """
        images = batch["image"]
        
        # Convert to tensor and normalize
        # Shape: (batch, H, W, C) -> (batch, C, H, W)
        tensors = torch.from_numpy(images).permute(0, 3, 1, 2).float() / 255.0
        tensors = self.transform(tensors).to(self.device)
        
        # Run inference
        with torch.no_grad():
            outputs = self.model(tensors)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            top_probs, top_indices = probabilities.topk(5, dim=1)
        
        # Get predictions
        predictions = []
        confidences = []
        
        for i in range(len(images)):
            top_class = top_indices[i, 0].item()
            predictions.append(self.labels[top_class])
            confidences.append(top_probs[i, 0].item())
        
        return {
            **batch,
            "prediction": np.array(predictions),
            "confidence": np.array(confidences),
        }

## 1.3 Run Batch Inference

In [ ]:
# Configure batch inference
BATCH_SIZE = 32
NUM_GPU_ACTORS = max(1, NUM_GPUS)

print(f"Running inference with {NUM_GPU_ACTORS} GPU actor(s)")
print(f"Batch size: {BATCH_SIZE}")
print("="*50)

In [ ]:
# Run batch inference
results_ds = image_ds.map_batches(
    ImageClassifier,
    batch_size=BATCH_SIZE,
    num_gpus=1 if NUM_GPUS > 0 else 0,
    concurrency=NUM_GPU_ACTORS,
)

# Show sample results
print("\nSample predictions:")
for row in results_ds.take(5):
    print(f"  Image {row['id']}: {row['prediction']} (confidence: {row['confidence']:.3f})")

---

# Part 2: Tabular Data Processing

Process large tabular datasets with ML models.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import pickle

## 2.1 Create Dataset and Train Model

In [ ]:
# Generate large synthetic dataset
X, y = make_classification(
    n_samples=100000,
    n_features=20,
    n_informative=10,
    random_state=42,
)

# Create DataFrame
feature_cols = [f"feature_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_cols)

print(f"Dataset shape: {df.shape}")

In [ ]:
# Train a simple model for demonstration
# In production, load your trained model
model = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)
model.fit(X[:1000], y[:1000])  # Train on small subset

# Save model
model_path = f"{OUTPUT_PATH}/rf_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f)

print(f"Model saved to {model_path}")

## 2.2 Define Sklearn Predictor

In [ ]:
class SklearnPredictor:
    """
    Predictor for scikit-learn models.
    
    Loads model once and runs on CPU workers.
    """
    
    def __init__(self, model_path: str):
        import pickle
        
        with open(model_path, "rb") as f:
            self.model = pickle.load(f)
        
        self.feature_cols = [f"feature_{i}" for i in range(20)]
    
    def __call__(self, batch: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
        """Run predictions on a batch."""
        # Extract features
        features = np.column_stack([batch[col] for col in self.feature_cols])
        
        # Get predictions and probabilities
        predictions = self.model.predict(features)
        probabilities = self.model.predict_proba(features)[:, 1]
        
        return {
            **batch,
            "prediction": predictions,
            "probability": probabilities,
        }

## 2.3 Run Distributed Inference

In [ ]:
# Convert to Ray Dataset
tabular_ds = ray.data.from_pandas(df)

print(f"Ray Dataset: {tabular_ds.count()} rows")

In [ ]:
# Run batch inference with sklearn model
# Uses CPU workers since sklearn doesn't benefit from GPU
tabular_results = tabular_ds.map_batches(
    SklearnPredictor,
    fn_constructor_kwargs={"model_path": model_path},
    batch_size=1000,
    num_cpus=1,
    concurrency=4,  # 4 parallel CPU workers
)

# Show sample results
print("\nSample predictions:")
for row in tabular_results.take(5):
    print(f"  Prediction: {row['prediction']}, Probability: {row['probability']:.3f}")

---

# Part 3: CPU → GPU Streaming Pipeline

Optimal pattern for heavy preprocessing + GPU inference.

In [ ]:
def preprocess_images(batch: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    """
    CPU-based preprocessing step.
    
    This runs on CPU workers while GPU workers run inference,
    maximizing overall throughput.
    """
    images = batch["image"]
    
    processed = []
    for img in images:
        # Example preprocessing operations
        # - Resize
        # - Augmentation
        # - Normalization
        
        # For demo, just add some noise
        noise = np.random.randn(*img.shape) * 10
        processed_img = np.clip(img + noise, 0, 255).astype(np.uint8)
        processed.append(processed_img)
    
    return {
        **batch,
        "image": np.array(processed),
        "preprocessed": np.ones(len(images), dtype=bool),
    }

In [ ]:
# Build streaming pipeline: CPU preprocess -> GPU inference
pipeline = (
    image_ds
    # Step 1: CPU preprocessing (runs on CPU workers)
    .map_batches(
        preprocess_images,
        batch_size=64,
        num_cpus=1,
        concurrency=2,
    )
    # Step 2: GPU inference (streams from CPU preprocessing)
    .map_batches(
        ImageClassifier,
        batch_size=BATCH_SIZE,
        num_gpus=1 if NUM_GPUS > 0 else 0,
        concurrency=NUM_GPU_ACTORS,
    )
)

print("Streaming pipeline created")
print("  1. CPU preprocessing (parallel)")
print("  2. GPU inference (streams from CPU)")

In [ ]:
# Execute pipeline
import time

start = time.time()
pipeline_results = list(pipeline.take(100))
elapsed = time.time() - start

print(f"\nProcessed {len(pipeline_results)} samples in {elapsed:.2f} seconds")
print(f"Throughput: {len(pipeline_results) / elapsed:.1f} samples/second")

---

# Part 4: Save Results

In [ ]:
# Save tabular results to Parquet (efficient columnar format)
tabular_output_path = f"{OUTPUT_PATH}/tabular_predictions"
tabular_results.write_parquet(tabular_output_path)
print(f"Saved tabular predictions to {tabular_output_path}")

In [ ]:
# Read back and verify
loaded_ds = ray.data.read_parquet(tabular_output_path)
print(f"\nLoaded {loaded_ds.count()} rows from Parquet")
print(f"Schema: {loaded_ds.schema()}")

## Performance Tips

In [ ]:
# Performance optimization tips
tips = """
1. BATCH SIZE
   - GPU: Larger batches (32-256) for better throughput
   - CPU: Moderate batches (100-1000) to avoid memory issues

2. CONCURRENCY
   - Set concurrency = number of GPUs for GPU workloads
   - Set concurrency = 2-4x CPU cores for CPU workloads

3. STREAMING
   - Ray Data streams data between stages automatically
   - No need to materialize intermediate results

4. MEMORY
   - Use object_store_memory in ray.init() if needed
   - Process in chunks with streaming execution

5. FILE FORMATS
   - Use Parquet for tabular data (columnar, compressed)
   - Use ray.data.read_images() for image directories
"""
print(tips)

## Cleanup

In [ ]:
shutdown_ray()
print("Ray cluster shutdown complete")

## Key Takeaways

1. **Stateful Predictors**: Load model once, reuse for all batches
2. **Heterogeneous Compute**: Use GPU for neural nets, CPU for sklearn
3. **Streaming Pipelines**: CPU preprocessing streams to GPU inference
4. **Scalability**: Same code works from 1 to 1000 GPUs
5. **Persistence**: Save to Parquet for efficient downstream use

## Next Steps

- **Image datasets**: Use `ray.data.read_images()` for real image files
- **Multi-modal**: Combine image, text, and tabular data
- **Online serving**: See GenAI cookbook for Ray Serve deployment

## Resources

- [Ray Data Documentation](https://docs.ray.io/en/latest/data/data.html)
- [Batch Inference Guide](https://docs.ray.io/en/latest/data/batch_inference.html)
- [Performance Tips](https://docs.ray.io/en/latest/data/performance-tips.html)